In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# §6.16 The Three-Dimensional Schrödinger Equation and Central Potentials

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.16",
    title="The Three-Dimensional Schrödinger Equation and Central Potentials",
    blurb="Angles and radius, separated. Any force that points toward a center lets "
    "the three-dimensional problem split cleanly: the angular part is a spherical "
    "harmonic we have already solved, and the radial part becomes a one-dimensional "
    "Schrödinger equation — with one new feature, a centrifugal barrier that angular "
    "momentum adds to the potential and that holds the particle away from the origin. "
    "Solve that 1D equation on the grid, and every central potential is within reach.",
    difficulty="advanced",
    estimate="160–195 min",
)

## Notebook overview

We now have both halves of the three-dimensional puzzle. Movement II taught us to solve *one-dimensional*
Schrödinger equations on a grid ([§6.10](schrodinger-on-a-computer.ipynb), [§6.11](bound-states-1d.ipynb)), and the last two notebooks built the *angular* machinery —
the algebra of angular momentum ([§6.14](angular-momentum-algebra.ipynb)) and its realization as the spherical harmonics on the sphere
([§6.15](orbital-angular-momentum.ipynb)). This notebook joins them, and the join is one of the most powerful moves in all of physics:
**separation of variables** for a **central potential**.

A central potential is one that depends only on the distance from a center, $V=V(r)$ — the Coulomb pull
of a nucleus, a spherical well, the isotropic oscillator, a screened nuclear force. Because such a
potential is rotationally symmetric, its Hamiltonian **commutes with $L^2$ and $L_z$** ([§6.15](orbital-angular-momentum.ipynb)), so by the
compatible-observables principle of [§6.6](pauli-uncertainty.ipynb) the energy eigenstates can be chosen to be simultaneous
eigenstates of $H$, $L^2$, and $L_z$. That forces every eigenstate to factor as
$\psi(r,\theta,\varphi)=R_{n,l}(r)\,Y_l^m(\theta,\varphi)$: a radial function times a spherical harmonic.
The angular factor is *already solved* — it is one of the $Y_l^m$ of [§6.15](orbital-angular-momentum.ipynb) — so all that remains is the
**radial function** $R(r)$.

And the radial problem turns out to be something we already know how to do. Substituting $u(r)=rR(r)$
removes the awkward first-derivative term and turns the radial equation into *exactly* a
one-dimensional Schrödinger equation on the half-line $r\ge0$,
$$-\frac{\hbar^2}{2m}u''+V_{\text{eff}}(r)\,u=E\,u,\qquad u(0)=0,$$
with just one new ingredient in the potential: the **centrifugal barrier** $\hbar^2 l(l+1)/2mr^2$, a
repulsive term that angular momentum adds to $V(r)$. It is the quantum image of the centrifugal effect —
it pushes higher-$l$ states outward, away from the center, and forces the wavefunction to vanish as
$u\sim r^{l+1}$ at the origin. With $V_{\text{eff}}$ in hand we simply hand the radial equation to the
**finite-difference eigensolver of [§6.10](schrodinger-on-a-computer.ipynb)** — the very same $(1,-2,1)/dx^2$ stencil, `numpy.linalg.eigh`,
now on the half-line — and read off the radial energies and wavefunctions of *any* central potential.

We check the machine against a case with an exact answer, the isotropic oscillator $E=(2n_r+l+\tfrac32)
\hbar\omega$, and notice a curiosity in passing: its energy depends only on the combination $2n_r+l$, so
different $(n_r,l)$ states pile up at the same energy — an "accidental" degeneracy that hints at a hidden
symmetry. The *same* phenomenon, in a sharper form, is the crown of the next notebook: hydrogen, whose
energy depends on $n=n_r+l+1$ but *not on $l$ at all*.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — the $u=rR$ reduction, the effective potential
$V+\hbar^2l(l+1)/2mr^2$, and the [§6.10](schrodinger-on-a-computer.ipynb) finite-difference `numpy.linalg.eigh` solver on a `numpy.linspace`
radial grid that avoids $r=0$.

> **Conventions and method notes.** $\hbar=m=1$ (and $\omega=1$ for the oscillator). The radial grid is
> the interior of $(0,r_{\max}]$ — we exclude $r=0$ (the centrifugal term diverges there) and $r_{\max}$,
> which imposes the Dirichlet walls $u(0)=u(r_{\max})=0$; $r_{\max}$ must be large enough that $u$ has
> decayed. Radial functions are normalized so $\int_0^\infty|u|^2dr=1$, equivalently $\int_0^\infty|R|^2
> r^2dr=1$ (since $u=rR$). The radial quantum number $n_r=0,1,2,\dots$ counts radial nodes. See Sakurai &
> Napolitano and Griffiths (central potentials, the radial equation); and Notebooks [§6.15](orbital-angular-momentum.ipynb) (the spherical
> harmonics, the angular part), [§6.10](schrodinger-on-a-computer.ipynb) (the finite-difference eigensolver, reused here), [§6.6](pauli-uncertainty.ipynb) (compatible
> observables $H,L^2,L_z$), and [§6.12](harmonic-oscillator.ipynb) (the 1-D oscillator, whose 3-D cousin appears below).

## Theory in brief

### Central potentials and rotational symmetry

A **central** potential depends only on $r=|\mathbf r|$, so

```{math}
:label: eq-central
H=-\frac{\hbar^2}{2m}\nabla^2+V(r),\qquad [H,L^2]=[H,L_z]=0 .
```

The Hamiltonian is rotationally invariant and commutes with $L^2$ and $L_z$ ([§6.15](orbital-angular-momentum.ipynb)). By the
compatible-observables principle ([§6.6](pauli-uncertainty.ipynb)), the energy eigenstates can be chosen as simultaneous eigenstates
of $H$, $L^2$, and $L_z$ — labelled by an energy, by $l$, and by $m$.

### Separation of variables

In spherical coordinates the Laplacian splits into a radial piece and an angular piece, and the angular
piece is exactly $-L^2/\hbar^2 r^2$ ([§6.15](orbital-angular-momentum.ipynb)). Substituting $\psi=R(r)Y_l^m(\theta,\varphi)$ and using
$L^2Y_l^m=\hbar^2 l(l+1)Y_l^m$,

```{math}
:label: eq-separation
\psi(r,\theta,\varphi)=R_{n,l}(r)\,Y_l^m(\theta,\varphi)\ \Longrightarrow\ -\frac{\hbar^2}{2m}\!\left(R''+\frac{2}{r}R'\right)+\left[V(r)+\frac{\hbar^2 l(l+1)}{2mr^2}\right]R=E\,R ,
```

the angular dependence cancels and the three-dimensional problem becomes a **family of one-dimensional
radial problems**, one for each $l$.

### The $u=rR$ reduction

The radial equation still carries a first-derivative term, but the substitution $u(r)=rR(r)$ removes it:

```{math}
:label: eq-radial
u(r)=rR(r)\ \Longrightarrow\ -\frac{\hbar^2}{2m}u''+V_{\text{eff}}(r)\,u=E\,u,\qquad u(0)=0 ,
```

which is *exactly* a one-dimensional Schrödinger equation on the half-line $r\ge0$. The boundary
condition $u(0)=0$ is regularity: $R=u/r$ must stay finite at the origin. Every technique of Movement II
applies unchanged.

### The centrifugal barrier

The effective potential carries one new term beyond $V(r)$:

```{math}
:label: eq-centrifugal
V_{\text{eff}}(r)=V(r)+\frac{\hbar^2 l(l+1)}{2mr^2} ,
```

the **centrifugal barrier** — a repulsive potential, growing with $l$, the quantum image of the classical
centrifugal effect. It pushes higher-$l$ states **outward** (the radial peak moves out with $l$) and
forces $u\sim r^{l+1}$ near the origin, so a particle with angular momentum avoids the center. Only $l=0$
($s$ states) feel no barrier and can reach $r=0$.

### Solving on the grid

Put $r$ on a grid that avoids $r=0$, build $V_{\text{eff}}$, and apply the [§6.10](schrodinger-on-a-computer.ipynb) finite-difference
eigenmethod:

```{math}
:label: eq-radial-grid
\text{any } V(r)\ \xrightarrow{\ u=rR,\ \text{stencil}+\mathrm{diag}\,V_{\text{eff}}+\texttt{eigh}\ }\ \{E_{n,l},\,u_{n,l}(r)\} .
```

This returns the radial energies $E_{n,l}$ and radial functions for *any* central $V(r)$; $n_r$ counts
the radial nodes, and the full state is $R_{n,l}(r)Y_l^m(\theta,\varphi)$.

### The isotropic oscillator, and a hint of hidden symmetry

For $V=\tfrac12 m\omega^2 r^2$ the spectrum is known exactly — in Cartesian coordinates the problem
separates into three 1-D oscillators of [§6.12](harmonic-oscillator.ipynb), and stacking their ladders gives, in the radial
labels $(n_r,l)$,

```{math}
:label: eq-3d-oscillator
E=(2n_r+l+\tfrac32)\,\hbar\omega ,
```

a clean check on the whole reduction. The energy depends only on the combination $2n_r+l$, so many
$(n_r,l)$ states share an energy — a degeneracy *beyond* the $(2l+1)$-fold one that rotational symmetry
guarantees. This "accidental" degeneracy signals a **hidden symmetry** (the oscillator's $SU(3)$); the
same phenomenon in hydrogen — where the Coulomb energy depends only on $n=n_r+l+1$, not on $l$ — is the
crown of [§6.17](hydrogen-atom.ipynb).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

HBAR = 1.0
MASS = 1.0


def effective_potential(r, V, l):
    r"""The effective radial potential $V_{\text{eff}}=V(r)+\hbar^2 l(l+1)/2mr^2$ {eq}`eq-centrifugal`.

    Adds the **centrifugal barrier** $\hbar^2 l(l+1)/2mr^2$ — repulsive, growing with $l$ — to the central
    potential ``V`` sampled on the grid ``r``. For $l=0$ there is no barrier.

    Parameters
    ----------
    r : numpy.ndarray
        The radial grid (must avoid $r=0$).
    V : numpy.ndarray
        The central potential $V(r)$ on the grid.
    l : int
        The orbital angular-momentum quantum number.

    Returns
    -------
    numpy.ndarray
        The effective potential on ``r``.
    """
    return V + HBAR**2 * l * (l + 1) / (2 * MASS * r**2)


def solve_radial(V, l, rmax, N):
    r"""Solve the radial Schrödinger equation for a central potential, via the §6.10 eigenmethod {eq}`eq-radial-grid`.

    Builds the interior grid $r\in(0,r_{\max})$ (excluding $r=0$ and $r_{\max}$, so $u(0)=u(r_{\max})=0$),
    forms $V_{\text{eff}}$ {eq}`eq-centrifugal`, and applies the **§6.10 finite-difference Hamiltonian** —
    the $(1,-2,1)/dr^2$ kinetic stencil (``numpy.diag``) plus $\mathrm{diag}\,V_{\text{eff}}$ — then
    diagonalizes with ``numpy.linalg.eigh``. Eigenvectors are divided by $\sqrt{dr}$ so $\int|u|^2dr=1$.

    Parameters
    ----------
    V : callable
        The central potential as a function ``V(r)`` of the radial grid.
    l : int
        The orbital angular-momentum quantum number.
    rmax : float
        The outer boundary (the box must be large enough that $u$ has decayed).
    N : int
        The number of interior grid points.

    Returns
    -------
    r : numpy.ndarray
        The radial grid.
    energies : numpy.ndarray
        The radial energies $E_{n_r,l}$, ascending.
    u : numpy.ndarray
        Column $n_r$ is the normalized radial function $u_{n_r,l}(r)=rR(r)$.
    """
    r = np.linspace(0.0, rmax, N + 2)[
        1:-1
    ]  # interior grid: r=0 and rmax excluded (Dirichlet walls)
    dr = r[1] - r[0]
    Veff = effective_potential(r, V(r), l)
    laplacian = (
        np.diag(-2 * np.ones(N))
        + np.diag(np.ones(N - 1), 1)
        + np.diag(np.ones(N - 1), -1)
    ) / dr**2
    H = -(HBAR**2 / (2 * MASS)) * laplacian + np.diag(
        Veff
    )  # the §6.10 Hamiltonian, now radial
    energies, u = np.linalg.eigh(H)
    return r, energies, u / np.sqrt(dr)


def radial_density(u, r):
    r"""The radial probability density $|u(r)|^2=r^2|R(r)|^2$ — the probability per unit $r$.

    Because $u=rR$, the quantity $|u|^2$ is already the radial probability density (with $\int|u|^2dr=1$):
    the probability of finding the particle in a shell $[r,r+dr]$, integrated over all angles.
    """
    return np.abs(u) ** 2


# a fine second-derivative helper for the analytic checks below (finite differences)
def _d1(f, h):
    return np.gradient(f, h)


def _d2(f, h):
    return np.gradient(np.gradient(f, h), h)

## Exercise 1 — Separation of variables

Show that a central potential separates: with $\psi=R(r)Y_l^m$, the three-dimensional Schrödinger
equation reduces to a one-dimensional **radial equation** in which $l$ enters only through
$\hbar^2 l(l+1)$ {eq}`eq-central`, {eq}`eq-separation`. We verify the separation by confirming
that an *exact* solution — the isotropic-oscillator radial ground state — satisfies the separated
radial equation.

1. Recall $H$ commutes with $L^2,L_z$ (rotational symmetry, [§6.15](orbital-angular-momentum.ipynb)), so eigenstates factor as
   $R(r)Y_l^m$ and the angular part contributes the c-number $\hbar^2 l(l+1)$ (from
   $L^2Y_l^m=\hbar^2 l(l+1) Y_l^m$).
2. Write the separated radial equation $-\frac{\hbar^2}{2m}(R''+\frac2r R')+[V+\frac{\hbar^2
   l(l+1)}{2mr^2}]R=ER$.
3. Take the known exact solution $R_{0,l}(r)\propto r^l e^{-r^2/2}$ of the isotropic oscillator
   and apply the radial operator (second derivatives by finite differences).
4. Confirm the result is $E\,R$ with $E=(l+\tfrac32)\hbar\omega$ — the separated equation holds.
   Frame: the angular problem is already solved ([§6.15](orbital-angular-momentum.ipynb)); only the radius remains.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    sep_ok,
    "a central potential separates: ψ=R(r)Yₗᵐ reduces H to a radial equation with l entering via ℏ²l(l+1), satisfied by the exact oscillator solution with E=(l+3/2)",
)

## Exercise 2 — The $u=rR$ reduction and the effective potential

Transform the radial equation into a clean one-dimensional Schrödinger equation with the
substitution $u(r)=rR(r)$, and identify the effective potential and its centrifugal barrier
{eq}`eq-radial`, {eq}`eq-centrifugal`.

1. Substitute $u=rR$; the key identity is $\frac{1}{r^2}(r^2R')'=\frac{u''}{r}$, which removes the
   first-derivative term.
2. Verify that identity numerically for a test $R(r)$, confirming the reduction to
   $-\frac{\hbar^2}{2m}u''+V_{\text{eff}}u=Eu$.
3. Read off $V_{\text{eff}}=V+\hbar^2 l(l+1)/ 2mr^2$ (the `effective_potential` helper) and note
   the boundary condition $u(0)=0$ (since $u=rR$).
4. Plot $V_{\text{eff}}$ for several $l$ to see the centrifugal barrier grow. Frame: the 3-D
   radial problem is *exactly* a 1-D problem — all of Movement II applies.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    reduction_ok,
    "the substitution u=rR removes the first-derivative term, giving a 1-D Schrödinger equation −(ℏ²/2m)u''+V_eff u=E u with V_eff=V+ℏ²l(l+1)/2mr² and u(0)=0",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Solving the radial equation on the grid

Solve the radial equation for a chosen central potential and given $l$ with the [§6.10](schrodinger-on-a-computer.ipynb)
finite-difference eigenmethod, and inspect the radial functions $u_{n_r,l}(r)$ and
$R_{n_r,l}(r)=u/r$ {eq}`eq-radial-grid`.

1. Set up the radial grid avoiding $r=0$ (`numpy.linspace`, interior points).
2. Build $V_{\text{eff}}$ and the finite-difference Hamiltonian — the $(1,-2,1)/dr^2$ stencil plus
   $\mathrm{diag} \,V_{\text{eff}}$ (the `solve_radial` helper, the [§6.10](schrodinger-on-a-computer.ipynb) engine).
3. Diagonalize with `numpy.linalg.eigh` for the radial energies and $u_{n_r,l}(r)$.
4. Plot $u$ (which vanishes at $r=0$) and $R=u/r$, and count the radial nodes ($n_r$). Frame: the
   [§6.10](schrodinger-on-a-computer.ipynb) solver, now on the half-line.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    grid_ok,
    "the finite-difference eigenmethod (§6.10) solves the radial equation: ascending energies, node-counting radial functions, and u(0)=0",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — The 3D isotropic oscillator: an exact check

Solve the isotropic oscillator $V=\tfrac12 m\omega^2 r^2$ and compare the radial spectrum to the
exact result $E=(2n_r+l+\tfrac32)\hbar\omega$, then note the degeneracy it implies
{eq}`eq-3d-oscillator`.

1. Solve the radial equation for $l=0,1,2$ with `solve_radial`.
2. Compare the lowest radial levels to $(2n_r+l+\tfrac32)\hbar\omega$ (`numpy.linalg.eigh` vs the
   formula).
3. Confirm agreement to grid accuracy.
4. Note the energy depends only on the combination $2n_r+l$, so different $(n_r,l)$ states share
   an energy — a degeneracy *beyond* the $(2l+1)$ that rotation requires. Frame: the reduction
   validated against an exact result, with a first hint of hidden symmetry.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    np.concatenate([levels[l] for l in [0, 1, 2]]),
    np.concatenate(
        [np.array([2 * nr + l + 1.5 for nr in range(3)]) for l in [0, 1, 2]]
    ),
    "the 3D isotropic oscillator spectrum is E=(2n_r+l+3/2)ℏω — the radial reduction validated against an exact result",
    atol=1e-2,
)

## Exercise 5 — The centrifugal barrier pushes states outward

Show that higher angular momentum moves the radial probability **outward** and suppresses the
wavefunction at the origin as $u\sim r^{l+1}$ {eq}`eq-centrifugal`.

1. For the oscillator, solve the lowest radial state ($n_r=0$) for $l=0,1,2,3$.
2. Find where the radial density $r^2|R|^2=|u|^2$ peaks (the `radial_density` helper,
   `numpy.argmax`).
3. Confirm the peak moves outward with $l$ (here to $r_{\text{peak}}=\sqrt{l+1}$).
4. Confirm $u\sim r^{l+1}$ near the origin (a `numpy.polyfit` of $\log u$ vs $\log r$ at small
   $r$). Frame: angular momentum holds the particle away from the center — the quantum centrifugal
   effect.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    outward,
    "the centrifugal barrier pushes higher-l states outward (the radial peak moves out with l) and suppresses them at the origin as u~r^(l+1)",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — A general central potential *(student)*

Solve a central potential with *no* elementary spectrum — a spherical finite well, and a screened
Coulomb (Yukawa) potential $-e^{-r/a}/r$ — and characterize its bound states, showing the method
needs only $V(r)$ {eq}`eq-radial-grid`.

1. Define $V(r)$ (a spherical well of depth $V_0$ and radius $a$; then the Yukawa potential).
2. Solve the radial equation for $l=0$ with `solve_radial`.
3. Report the bound energies ($E<0$) and count them.
4. Note how the count depends on the potential's depth and range (a callback to the 1-D well
   counting of [§6.11](bound-states-1d.ipynb)). Frame: the method needs only $V(r)$ — every central problem is accessible.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    student_ok,
    "the radial eigenmethod solves arbitrary central potentials: it returns sensible bound states for a spherical well and a screened Coulomb (Yukawa) potential",
)

## Exercise 7 — One equation for every central force

Rotational symmetry did most of the work for us. Because a central potential commutes with the angular
momentum ([§6.6](pauli-uncertainty.ipynb), [§6.15](orbital-angular-momentum.ipynb)), every eigenstate is a spherical harmonic times a radial function, and the radial
function obeys a one-dimensional Schrödinger equation we already knew how to solve ([§6.10](schrodinger-on-a-computer.ipynb)). The only new
ingredient was the **centrifugal barrier**, the repulsive term angular momentum adds to the potential,
which pushes orbiting states outward and holds them off the center. This single reduction handles the
spherical well, the isotropic oscillator, and the screened nucleus alike — and, next, the one central
potential that matters most.

**This exercise (synthesis).** No new computation: the reduction is the result. Three dimensions sounded
harder than one, and for a *general* potential it is. But symmetry is a solvent: point the force at a
center, and the hard three-dimensional problem dissolves into the spherical harmonics we already had
([§6.15](orbital-angular-momentum.ipynb)) and a radial equation we already solved ([§6.10](schrodinger-on-a-computer.ipynb)). Almost every quantum problem you can actually
compute — atoms, nuclei, quantum dots — is a central-force problem, for exactly this reason. In [§6.17](hydrogen-atom.ipynb) the
same radial equation, now with the Coulomb potential $V=-e^2/4\pi\varepsilon_0 r$, gives the **hydrogen
atom**: its spectrum, its orbitals, and the deepest surprise of the volume — why its energy does not
depend on $l$ at all, the sharp version of the "accidental" oscillator degeneracy we glimpsed here.

## Notebook summary

The three-dimensional Schrödinger equation, reduced to a radial one — the third notebook of Movement III.

- **Rotational symmetry** {eq}`eq-central`: a central $V(r)$ makes $H$ commute with $L^2,L_z$ ([§6.6](pauli-uncertainty.ipynb), [§6.15](orbital-angular-momentum.ipynb)),
  so eigenstates factor as $\psi=R_{n,l}(r)Y_l^m(\theta,\varphi)$.
- **Separation** {eq}`eq-separation`: the angular part is a solved spherical harmonic; the radial part is
  an ODE with $l$ entering only via $\hbar^2 l(l+1)$.
- **The $u=rR$ reduction** {eq}`eq-radial`: turns the radial equation into a 1-D Schrödinger equation on
  the half-line, $-\frac{\hbar^2}{2m}u''+V_{\text{eff}}u=Eu$, $u(0)=0$.
- **The centrifugal barrier** {eq}`eq-centrifugal`: $V_{\text{eff}}=V+\hbar^2 l(l+1)/2mr^2$ pushes
  higher-$l$ states outward ($r_{\text{peak}}$ grows with $l$) and forces $u\sim r^{l+1}$ at the origin.
- **The grid method** {eq}`eq-radial-grid`: the [§6.10](schrodinger-on-a-computer.ipynb) stencil $+\,\mathrm{diag}\,V_{\text{eff}}+$ `eigh`
  solves *any* central $V(r)$ — verified on the oscillator, a spherical well, and a Yukawa potential.
- **Exact check** {eq}`eq-3d-oscillator`: $E=(2n_r+l+\tfrac32)\hbar\omega$; its $2n_r+l$ degeneracy hints
  at a hidden symmetry — the parallel to hydrogen's $l$-degeneracy ([§6.17](hydrogen-atom.ipynb)).

Point a force at a center and three dimensions collapse to one. The radial equation is ready; next, the
Coulomb potential turns it into the hydrogen atom.

## Outlook

- **The hydrogen atom ([§6.17](hydrogen-atom.ipynb))**: the Coulomb radial equation, the Rydberg spectrum $-13.6/n^2\,$eV, the
  orbitals, and the $l$-degeneracy / hidden symmetry — the crown of the volume.
- **Spin added to the orbital picture ([§6.18](spin-magnetic.ipynb))**: the full set of quantum numbers $n,l,m,m_s$.
- **Many-electron atoms and the periodic table** (a horizon; the angular structure of [§6.15](orbital-angular-momentum.ipynb) with this
  radial structure, plus the exclusion principle).
- **The hidden symmetries of the Coulomb and oscillator potentials** (the Runge–Lenz vector; named in
  [§6.17](hydrogen-atom.ipynb)).
- **Cross-reference** [§6.15](orbital-angular-momentum.ipynb) (the spherical harmonics, the angular part), [§6.10](schrodinger-on-a-computer.ipynb) (the eigensolver, reused),
  [§6.6](pauli-uncertainty.ipynb) (compatible observables), [§6.12](harmonic-oscillator.ipynb) (the 1-D oscillator), and forward to [§6.17](hydrogen-atom.ipynb), [§6.18](spin-magnetic.ipynb).

In [ ]:
from ecp.style import footer

footer()